# Foil

The {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend` includes special utilities for working with foil-sealed plates, specifically:

1. a function to pierce foil before aspirating from the plate, and
2. a function to keep the plate down while moving the channels up to avoid lifting the plate.

## Example setup

```{note}
While this example uses high volume tips, it _might_ be possible to use other tip types to pierce the foil. However, 50uL tips are very soft and probably can't be used.
```

In [1]:
from pylabrobot.hamilton.liquid_handlers.star import STAR
from pylabrobot.resources.hamilton import STARLetDeck
from pylabrobot.resources import (
  TIP_CAR_480_A00,
  PLT_CAR_L5AC_A00,
  hamilton_96_tiprack_1000uL_filter,
  AGenBio_4_troughplate_75000uL_Vb,
)

deck = STARLetDeck()
star = STAR(deck=deck)
await star.setup()

# assign a tip rack
tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
tip_carrier[0] = tip_rack = hamilton_96_tiprack_1000uL_filter(name="tip_rack")
deck.assign_child_resource(tip_carrier, rails=3)

# assign a plate
plt_carrier = PLT_CAR_L5AC_A00(name="plt_carrier")
plt_carrier[0] = plate = AGenBio_4_troughplate_75000uL_Vb(name="plate")
deck.assign_child_resource(plt_carrier, rails=10)

2026-04-02 17:42:01,004 - pylabrobot.io.usb - INFO - Finding USB device...
2026-04-02 17:42:01,008 - pylabrobot.io.usb - INFO - Found USB device.
2026-04-02 17:42:01,008 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0


## Breaking the foil before using a plate

It is important to break the foil before aspirating because tiny foil pieces can get stuck in the tip, drastically changing the liquid handling characteristics.

In this example, we will use an 8 channel workcell and use the inner 6 channels for breaking the foil and then aspirating. We will use the outer 2 channels to keep the plate down while the inner channels are moving up.

In [2]:
well = plate.get_well("A1")
await star.pip.pick_up_tips(tip_rack["A1:H1"])

In [3]:
aspiration_channels = [1, 2, 3, 4, 5, 6]
hold_down_channels = [0, 7]
await star.driver.pip.pierce_foil(
    wells=[well],
    piercing_channels=aspiration_channels,
    hold_down_channels=hold_down_channels,
    move_inwards=4,
    deck=deck,
    one_by_one=False,
)

In [4]:
await star.pip.drop_tips(tip_rack["A1:H1"])

![gif of piercing foil](img/pierce_foil.gif)

## Holding the plate down

Holding the plate down while moving channels up after aspiration consists of two parts:
1. Making the channels stay down after a liquid handling operation has finished. By default, STAR will move channels up to traversal height.
2. Putting two channels on the edges of the plate to hold it down, while moving the other channels up.

In [6]:
await star.pip.return_tips()

In [7]:
await star.pip.pick_up_tips(tip_rack["A2:H2"])

In [8]:
from pylabrobot.hamilton.liquid_handlers.star.pip_backend import STARPIPBackend

num_channels = len(aspiration_channels)
await star.pip.aspirate(
  [well] * num_channels,
  vols=[100] * num_channels,
  use_channels=aspiration_channels,
  backend_params=STARPIPBackend.AspirateParams(
    min_z_endpos=well.get_location_wrt(deck, z="cavity_bottom").z,
    surface_following_distance=[0] * num_channels,
    pull_out_distance_transport_air=[0] * num_channels,
  ),
)

await star.driver.pip.step_off_foil(
  well,
  front_channel=7,
  back_channel=0,
  deck=deck,
  move_inwards=5,
)

In [9]:
await star.pip.drop_tips(tip_rack["A2:H2"])

![gif of holding down foil](img/step_off_foil.gif)

---

<iframe width="640" height="360" src="https://www.youtube.com/embed/urglg3WimHA" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>